# Phase 2B.2 - Context Depth 3 (Colab)

Run both selected prompt finalists at one context depth on the fixed 80/20 subsets. Depth 5 is reused from Phase 2B.1.


In [ ]:
from pathlib import Path
REPO_URL='https://github.com/ThomasdeCarpio/Text-Mining---NewsQA-RAG.git'
REPO_COMMIT='973e1a7758de1defeda7f88d0cb60e34f57eb51f'
PLATFORM='colab'
RUN_ID='phase2b_2_depth_3'
PREPARATION_BUNDLE_PATH=''  # Required except in Phase 2B.0; Drive path on Colab or attached Kaggle input.
BASELINE_RESULTS_PATH=''  # Phase 2B.0 only: attached Phase 2A full-results ZIP/directory.
RESTORE_CHECKPOINT_PATH=''  # Optional checkpoint from the same notebook only.
EXECUTE_API_CALLS=False  # Set True only for an approved execution notebook.
GEMINI_SECRET_NAME='GEMINI_API_KEY_1'  # Give each Colab prompt/depth notebook its own key.
UPSTREAM_DECISION_PATH=''  # Optional JSON decision record; retained in provenance.
RUN_STAGE=RUN_ID
PROMPT_ID=None
PROMPT_FINALISTS=[]
FINALIST_CONFIG=None
FINALIST_CONFIGS=[]
LOCKED_WINNER=None
HELDOUT_APPROVED=False
CONTEXT_DEPTH=None
PROMPT_FINALISTS=[]  # Set exactly two IDs selected from the three Phase 2B.1 results.
CONTEXT_DEPTH=3
SEED=42
SCREENING_QUESTIONS=80
JUDGE_CALIBRATION_QUESTIONS=20
FINAL_HELDOUT_ARTICLES=50
GENERATOR_MODEL='gemini-3.1-flash-lite'
GENERATOR_REASONING_EFFORT='minimal'
GENERATOR_MAX_TOKENS=512
GENERATOR_MIN_INTERVAL_SECONDS=4.2
JUDGE_MODEL='accounts/fireworks/models/glm-5p3-flash'
JUDGE_REASONING_EFFORT='low'
JUDGE_MAX_TOKENS=2048
GENERATOR_INPUT_PER_MILLION_USD=0.25
GENERATOR_OUTPUT_PER_MILLION_USD=1.50
JUDGE_INPUT_PER_MILLION_USD=0.15
JUDGE_OUTPUT_PER_MILLION_USD=0.50
TOP_K=20
RERANK_TOP_N=5


## 1. Environment and immutable inputs

Set only the configuration values at the top. Every run validates the locked artifact, preparation bundle, subset hashes, and repository commit.


In [ ]:
import hashlib, json, os, shutil, string, subprocess, sys, time, zipfile
from google.colab import drive, userdata
drive.mount('/content/drive')
KAGGLE_INPUT=Path('/content/drive/MyDrive'); RUNTIME_ROOT=Path('/content')
def optional_secret(name):
    try: return userdata.get(name) or ''
    except Exception: return ''
PROJECT_ROOT=RUNTIME_ROOT/'Text-Mining---NewsQA-RAG'
WORK_ROOT=RUNTIME_ROOT/RUN_ID
DATA_ROOT=WORK_ROOT/'data'; INDEX_ROOT=WORK_ROOT/'index'; BASELINE_ROOT=WORK_ROOT/'baseline'
RUNS_ROOT=WORK_ROOT/'runs'; IDS_ROOT=WORK_ROOT/'question_ids'; PROMPT_ROOT=WORK_ROOT/'prompts'; RESULTS=WORK_ROOT/'results'; LOGS=WORK_ROOT/'logs'
prep_input=Path(PREPARATION_BUNDLE_PATH) if PREPARATION_BUNDLE_PATH else None
if prep_input is None and PLATFORM=='kaggle':
    candidates=sorted(KAGGLE_INPUT.rglob('phase2b_preparation_bundle.zip'))
    prep_input=candidates[-1] if candidates else None
assert prep_input and prep_input.exists(), 'Attach/set PREPARATION_BUNDLE_PATH from Phase 2B.0'
WORK_ROOT.mkdir(parents=True,exist_ok=True); shutil.unpack_archive(prep_input,WORK_ROOT)
checkpoint_input=Path(RESTORE_CHECKPOINT_PATH) if RESTORE_CHECKPOINT_PATH else None
if checkpoint_input and checkpoint_input.exists():
    WORK_ROOT.mkdir(parents=True,exist_ok=True); shutil.unpack_archive(checkpoint_input,WORK_ROOT); print('Restored checkpoint:',checkpoint_input)
for path in [DATA_ROOT,INDEX_ROOT,BASELINE_ROOT,RUNS_ROOT,IDS_ROOT,PROMPT_ROOT,RESULTS,LOGS]: path.mkdir(parents=True,exist_ok=True)
assert not REPO_COMMIT.startswith('SET_TO_'), 'Pin REPO_COMMIT after committing the split notebooks'
if not PROJECT_ROOT.exists(): subprocess.run(['git','clone','--filter=blob:none',REPO_URL,str(PROJECT_ROOT)],check=True)
subprocess.run(['git','fetch','--depth=1','origin',REPO_COMMIT],cwd=PROJECT_ROOT,check=True,timeout=180)
subprocess.run(['git','checkout','--detach',REPO_COMMIT],cwd=PROJECT_ROOT,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt'],cwd=PROJECT_ROOT,check=True)
GENERATOR_API_KEY=optional_secret(GEMINI_SECRET_NAME); JUDGE_API_KEY=optional_secret('FIREWORKS_API_KEY'); HF_TOKEN=optional_secret('HF_TOKEN')
os.environ.update({'HF_HOME':str(RUNTIME_ROOT/'hf_cache'),'TOKENIZERS_PARALLELISM':'false','OMP_NUM_THREADS':'1','MKL_NUM_THREADS':'1','PYTHONUNBUFFERED':'1','LANGCHAIN_TRACING_V2':'false','LANGSMITH_TRACING':'false'})
if EXECUTE_API_CALLS:
    assert GENERATOR_API_KEY, f'Configure the secret named {GEMINI_SECRET_NAME}'
    assert JUDGE_API_KEY, 'Configure FIREWORKS_API_KEY'
print('Run:',RUN_ID,'| platform:',PLATFORM,'| API execution:',EXECUTE_API_CALLS)


In [ ]:
import pandas as pd, yaml
from IPython.display import display
def sha256_file(path,block_size=1024*1024):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(block_size),b''): digest.update(block)
    return digest.hexdigest()
def write_json(path,value): Path(path).write_text(json.dumps(value,indent=2,sort_keys=True)+'\n',encoding='utf-8')
def load_jsonl(path): return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]
def write_jsonl(path,records): Path(path).write_text(''.join(json.dumps(row,sort_keys=True)+'\n' for row in records),encoding='utf-8')
def write_checkpoint():
    checkpoint=RUNTIME_ROOT/f'{RUN_ID}_checkpoint.zip'; temporary=checkpoint.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED) as archive:
        for name in ['baseline','index','runs','question_ids','prompts','results','logs','heldout_trace']:
            root=WORK_ROOT/name
            if root.exists():
                for path in root.rglob('*'):
                    if path.is_file(): archive.write(path,path.relative_to(WORK_ROOT))
    temporary.replace(checkpoint); print('Checkpoint:',checkpoint,round(checkpoint.stat().st_size/2**20,1),'MiB',flush=True); return checkpoint
def run_command(command,label,env_overrides=None):
    command=[str(value) for value in command]; log_path=LOGS/f'{label}_{time.strftime("%Y%m%d_%H%M%S")}.log'
    print('$',' '.join(command),flush=True); print('Log:',log_path,flush=True)
    with log_path.open('w',encoding='utf-8') as log:
        env=os.environ.copy(); env.update(env_overrides or {})
        process=subprocess.Popen(command,cwd=PROJECT_ROOT,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1)
        for line in process.stdout: print(line,end='',flush=True); log.write(line); log.flush()
        code=process.wait()
    if code: write_checkpoint(); raise subprocess.CalledProcessError(code,command)
    return log_path
def stable_key(seed,*parts): return hashlib.sha256((':'.join([str(seed),*map(str,parts)])).encode()).hexdigest()
def configuration_fingerprint(prompt_id,context_depth,ids):
    payload={'baseline_fingerprint':baseline_manifest['run_fingerprint'],'prompt_id':prompt_id,'prompt':prompt_registry[prompt_id]['system_prompt'],'context_depth':context_depth,'question_ids':list(ids),'generator_model':GENERATOR_MODEL,'generator_reasoning_effort':GENERATOR_REASONING_EFFORT}
    return hashlib.sha256(json.dumps(payload,sort_keys=True,separators=(',',':')).encode()).hexdigest()
def article_balanced_order(rows,seed):
    buckets={}
    for row in rows: buckets.setdefault(row['article_key'],[]).append(row)
    for article in buckets: buckets[article].sort(key=lambda row:stable_key(seed,row['question_id']))
    articles=sorted(buckets,key=lambda article:stable_key(seed,article)); ordered=[]
    while any(buckets.values()):
        for article in articles:
            if buckets[article]: ordered.append(buckets[article].pop(0))
    return ordered
def question_type(question):
    first=question.strip().lower().split(maxsplit=1)[0].strip(string.punctuation) if question.strip() else 'other'
    return first if first in {'who','what','when','where','why','how','which'} else ('yes_no' if first in {'is','are','was','were','do','does','did','has','have','had','can','could','will','would'} else 'other')
def proportional_stratified_sample(rows,n,seed):
    assert 0<n<=len(rows); groups={}
    for row in rows: groups.setdefault((row['question_type'],row['gold_in_top5']),[]).append(row)
    quotas={key:min(len(group),int(n*len(group)/len(rows))) for key,group in groups.items()}
    while sum(quotas.values())<n:
        candidates=[key for key,group in groups.items() if quotas[key]<len(group)]
        key=max(candidates,key=lambda value:(n*len(groups[value])/len(rows)-quotas[value],stable_key(seed,*value)))
        quotas[key]+=1
    selected=[]
    for key,group in groups.items():
        ordered=article_balanced_order(group,seed)
        selected.extend(ordered[:quotas[key]])
    return [row['question_id'] for row in sorted(selected,key=lambda row:stable_key(seed,row['question_id']))]


## 2. Load validated inputs

The Phase 2B.0 bundle supplies the frozen baseline traces and subset manifest. The locked corpus artifact is downloaded independently and verified by SHA-256.


In [ ]:
from huggingface_hub import hf_hub_download
artifact_root=DATA_ROOT/'locked-bge-m3-512-64-deduplicated-v2'
if not (artifact_root/'bundle_manifest.json').exists():
    bundle=hf_hub_download(repo_id=HF_ARTIFACT_REPO_ID,repo_type='dataset',revision=HF_ARTIFACT_REVISION,filename=HF_ARTIFACT_FILENAME,token=HF_TOKEN or None)
    assert sha256_file(bundle)==HF_ARTIFACT_SHA256
    shutil.unpack_archive(bundle,artifact_root)
bundle_manifest=json.loads((artifact_root/'bundle_manifest.json').read_text())
assert bundle_manifest['statistics']['chunks']==22766 and bundle_manifest['statistics']['resolved_questions']==1152
for relative,record in bundle_manifest['artifacts'].items():
    path=artifact_root/relative; assert path.exists() and path.stat().st_size==record['bytes'] and sha256_file(path)==record['sha256'], relative
testset=artifact_root/'testset_resolved.jsonl'; chunks=artifact_root/'chunks.jsonl'; sparse_index=artifact_root/'bge_m3_sparse.pkl'
assert BASELINE_ROOT.exists(), 'Preparation bundle does not contain baseline inputs'
manifest_candidates=list(BASELINE_ROOT.rglob('run_manifest.json')); assert manifest_candidates, 'Baseline run_manifest.json is missing'
baseline_manifest_path=next(path for path in manifest_candidates if len(json.loads(path.read_text()).get('inputs',{}).get('question_ids',[]))==281)
baseline_dir=baseline_manifest_path.parent; baseline_manifest=json.loads(baseline_manifest_path.read_text())
baseline_retrievals=baseline_dir/'retrievals.jsonl'; baseline_predictions=baseline_dir/'predictions.jsonl'; baseline_judges=baseline_dir/'judge_results.jsonl'
for path in [baseline_retrievals,baseline_predictions,baseline_judges]: assert path.exists(), path
assert baseline_manifest['inputs']['generator_model']==GENERATOR_MODEL
assert baseline_manifest['inputs'].get('generator_reasoning_effort')==GENERATOR_REASONING_EFFORT
assert baseline_manifest['inputs']['rerank_top_n']==RERANK_TOP_N
print('Imported baseline:',baseline_dir); print('Development questions:',len(baseline_manifest['inputs']['question_ids']))


In [ ]:
config=yaml.safe_load((PROJECT_ROOT/'configs/config.yaml').read_text())
config['chunking'].update({'strategy':'recursive','chunk_size':512,'chunk_overlap':64})
config['llm'].update({'model':GENERATOR_MODEL,'temperature':0.0,'max_tokens':GENERATOR_MAX_TOKENS,'reasoning_effort':GENERATOR_REASONING_EFFORT})
config['retrieval'].update({'retriever':'sparse','top_k':TOP_K})
config['retrieval']['sparse'].update({'method':'bge-m3','model':'BAAI/bge-m3','device':'cuda'})
config['retrieval']['reranker'].update({'enabled':True,'type':'cross-encoder','model':'BAAI/bge-reranker-large','top_n':RERANK_TOP_N,'batch_size':8,'device':'cuda'})
config_path=INDEX_ROOT/'phase2b_config.yaml'; config_path.write_text(yaml.safe_dump(config,sort_keys=False),encoding='utf-8')
profile=json.loads((artifact_root/'deduplication/deduplicated.variant.json').read_text())
config_hash=hashlib.sha256(json.dumps(config,sort_keys=True,separators=(',',':')).encode()).hexdigest()
profile['pipeline'].update({'config_path':str(config_path),'config_sha256':config_hash})
profile['database'].update({'indexed':False,'chunk_count':22766})
profile['artifacts']['chunks']={'path':str(chunks),'bytes':chunks.stat().st_size,'sha256':sha256_file(chunks)}
profile['artifacts']['testset_resolved']={'path':str(testset),'bytes':testset.stat().st_size,'sha256':sha256_file(testset)}
profile['artifacts']['bm25']={'path':str(sparse_index),'bytes':sparse_index.stat().st_size,'sha256':sha256_file(sparse_index)}
profile_path=INDEX_ROOT/'phase2b_variant.json'; write_json(profile_path,profile)
prompt_registry=yaml.safe_load((PROJECT_ROOT/'configs/experiments/phase2_generation_prompts.yaml').read_text())['prompts']
from newsqa_rag.llm import OpenAILLM
assert set(prompt_registry)=={'p0','p1','p2','p3'} and prompt_registry['p0']['system_prompt']==OpenAILLM.DEFAULT_SYSTEM_PROMPT
for prompt_id,record in prompt_registry.items(): (PROMPT_ROOT/f'{prompt_id}.txt').write_text(record['system_prompt'],encoding='utf-8')
display(pd.DataFrame([{'prompt_id':key,'name':value['name'],'hypothesis':value['hypothesis']} for key,value in prompt_registry.items()]))


In [ ]:
prepared_subset_path=RESULTS/'subset_manifest.json'
assert prepared_subset_path.exists(), 'Preparation bundle is missing subset_manifest.json'
prepared_subset_manifest=json.loads(prepared_subset_path.read_text())
test_rows={row['question_id']:row for row in load_jsonl(testset)}
trace_records={row['question_id']:row for row in load_jsonl(baseline_retrievals)}
development_ids=baseline_manifest['inputs']['question_ids']; assert len(development_ids)==281 and len(set(development_ids))==281
assert all(qid in test_rows and qid in trace_records and trace_records[qid]['status']=='success' for qid in development_ids)
development_articles={test_rows[qid]['article_key'] for qid in development_ids}; assert len(development_articles)==50
heldout_pool_articles=sorted({row['article_key'] for row in test_rows.values()}-development_articles); assert len(heldout_pool_articles)==150
heldout_pool_ids=[qid for qid,row in test_rows.items() if row['article_key'] in set(heldout_pool_articles)]; assert len(heldout_pool_ids)==871
heldout_articles=sorted(heldout_pool_articles,key=lambda article:stable_key(SEED+4,article))[:FINAL_HELDOUT_ARTICLES]
heldout_article_set=set(heldout_articles); heldout_ids=[qid for qid,row in test_rows.items() if row['article_key'] in heldout_article_set]
heldout_reserve_ids=[qid for qid in heldout_pool_ids if qid not in set(heldout_ids)]
assert len(heldout_articles)==50 and len(heldout_ids)==284 and len(heldout_reserve_ids)==587
sampling_rows=[]
for qid in development_ids:
    row=test_rows[qid]; ranked=trace_records[qid]['trace']['retrieved_ids'][:5]
    sampling_rows.append({'question_id':qid,'article_key':row['article_key'],'question_type':question_type(row['question']),'gold_in_top5':bool(set(row['relevant_chunk_ids'])&set(ranked))})
smoke_ids=proportional_stratified_sample(sampling_rows,5,SEED+1)
screening_ids=proportional_stratified_sample(sampling_rows,SCREENING_QUESTIONS,SEED+2)
screening_rows=[row for row in sampling_rows if row['question_id'] in set(screening_ids)]
judge_calibration_ids=proportional_stratified_sample(screening_rows,JUDGE_CALIBRATION_QUESTIONS,SEED+3)
subsets={'smoke':smoke_ids,'screening':screening_ids,'judge_calibration':judge_calibration_ids,'development':development_ids,'heldout':heldout_ids,'heldout_reserve':heldout_reserve_ids}
for name,ids in subsets.items(): write_json(IDS_ROOT/f'{name}.json',ids)
subset_manifest={'schema_version':1,'seed':SEED,'source_baseline_fingerprint':baseline_manifest['run_fingerprint'],'counts':{name:len(ids) for name,ids in subsets.items()},'sha256':{name:sha256_file(IDS_ROOT/f'{name}.json') for name in subsets},'heldout_selection':{'method':'seeded_article_sample','seed':SEED+4,'articles':len(heldout_articles),'questions':len(heldout_ids),'article_ids':heldout_articles},'heldout_outputs_accessed_for_selection':False}
write_json(RESULTS/'subset_manifest.json',subset_manifest); display(pd.DataFrame([{'subset':name,'questions':len(ids)} for name,ids in subsets.items()]))
display(pd.DataFrame(sampling_rows).groupby(['question_type','gold_in_top5']).size().rename('development').reset_index())
assert subset_manifest['counts']==prepared_subset_manifest['counts'], 'Subset counts differ from Phase 2B.0'
assert subset_manifest['sha256']==prepared_subset_manifest['sha256'], 'Subset IDs differ from Phase 2B.0'
assert subset_manifest['heldout_selection']==prepared_subset_manifest['heldout_selection'], 'Held-out article sample differs from Phase 2B.0'
print('Preparation subset hashes verified')


In [ ]:
def filter_records(source,ids):
    wanted=set(ids); return [row for row in load_jsonl(source) if row.get('question_id') in wanted]
def materialize_p0(ids,label,judge_ids):
    run_dir=RUNS_ROOT/f'{label}__p0__d5'; run_dir.mkdir(parents=True,exist_ok=True)
    manifest=json.loads(baseline_manifest_path.read_text()); manifest['n_questions']=len(ids); manifest['inputs']['question_ids']=list(ids); manifest['status']='complete'; manifest['derived_subset_of']=baseline_manifest['run_fingerprint']
    write_json(run_dir/'run_manifest.json',manifest)
    for name,source,selected in [('retrievals.jsonl',baseline_retrievals,ids),('predictions.jsonl',baseline_predictions,ids),('judge_results.jsonl',baseline_judges,judge_ids)]: write_jsonl(run_dir/name,filter_records(source,selected))
    write_json(run_dir/'tuning_config.json',{'stage':label,'prompt_id':'p0','context_depth':5,'configuration_fingerprint':configuration_fingerprint('p0',5,ids),'reused_phase2a':True})
    run_command([sys.executable,'-u','scripts/score_benchmark_predictions.py','--run-dir',run_dir],f'score_{label}_p0_d5')
    return run_dir
def run_variant(prompt_id,context_depth,ids,label,judge_ids):
    assert prompt_id in prompt_registry and context_depth in {1,3,5}
    if prompt_id=='p0' and context_depth==5 and set(ids)<=set(development_ids): return materialize_p0(ids,label,judge_ids)
    assert EXECUTE_API_CALLS, 'Set EXECUTE_API_CALLS=True only when this registered stage is approved'
    run_dir=RUNS_ROOT/f'{label}__{prompt_id}__d{context_depth}'; ids_path=IDS_ROOT/f'active_{label}_{prompt_id}_d{context_depth}.json'; write_json(ids_path,list(ids))
    command=[sys.executable,'-u','scripts/collect_benchmark_predictions.py','--retriever','sparse','--reranker','cross-encoder','--reranker-model','BAAI/bge-reranker-large','--testset',testset,'--variant-manifest',profile_path,'--config',config_path,'--run-dir',run_dir,'--question-ids-file',ids_path,'--top-k',TOP_K,'--rerank-top-n',RERANK_TOP_N,'--generator-model',GENERATOR_MODEL,'--prompt-id',prompt_id,'--system-prompt-file',PROMPT_ROOT/f'{prompt_id}.txt','--context-depth',context_depth,'--source-retrievals',baseline_retrievals,'--generation-min-interval-seconds',GENERATOR_MIN_INTERVAL_SECONDS,'--max-attempts',3,'--retry-failed','--progress']
    run_command(command,f'generate_{label}_{prompt_id}_d{context_depth}',{'GEMINI_API_KEY':GENERATOR_API_KEY})
    run_command([sys.executable,'-u','scripts/score_benchmark_predictions.py','--run-dir',run_dir],f'score_prejudge_{label}_{prompt_id}_d{context_depth}')
    judge_ids_path=IDS_ROOT/f'judge_{label}_{prompt_id}_d{context_depth}.json'; write_json(judge_ids_path,list(judge_ids))
    judge=[sys.executable,'-u','scripts/judge_benchmark_predictions.py','--run-dir',run_dir,'--judge-provider','fireworks','--judge-model',JUDGE_MODEL,'--reasoning-effort',JUDGE_REASONING_EFFORT,'--judge-max-tokens',JUDGE_MAX_TOKENS,'--question-ids-file',judge_ids_path,'--batch-size',1,'--max-workers',1,'--max-attempts',3,'--retry-failed','--require-complete-metrics','--progress']
    run_command(judge,f'judge_{label}_{prompt_id}_d{context_depth}',{'FIREWORKS_API_KEY':JUDGE_API_KEY})
    run_command([sys.executable,'-u','scripts/score_benchmark_predictions.py','--run-dir',run_dir],f'score_{label}_{prompt_id}_d{context_depth}')
    run_fingerprint=json.loads((run_dir/'run_manifest.json').read_text())['run_fingerprint']
    write_json(run_dir/'tuning_config.json',{'stage':label,'prompt_id':prompt_id,'context_depth':context_depth,'configuration_fingerprint':configuration_fingerprint(prompt_id,context_depth,ids),'collector_run_fingerprint':run_fingerprint,'prompt_sha256':sha256_file(PROMPT_ROOT/f'{prompt_id}.txt'),'generator_reasoning_effort':GENERATOR_REASONING_EFFORT,'judge_reasoning_effort':JUDGE_REASONING_EFFORT,'reused_phase2a':False})
    return run_dir
def execute_matrix(configurations,ids,label,judge_ids):
    outputs=[]
    for config_record in configurations: outputs.append(run_variant(config_record['prompt_id'],config_record['context_depth'],ids,label,judge_ids))
    return outputs


## 3. Execute this assigned stage

The output is resumable. Do not change configuration after successful records exist.


In [ ]:
assert len(PROMPT_FINALISTS)==2 and len(set(PROMPT_FINALISTS))==2
assert set(PROMPT_FINALISTS)<={'p0','p1','p2','p3'}
assert CONTEXT_DEPTH in {1,3}
assert EXECUTE_API_CALLS, 'Set EXECUTE_API_CALLS=True after recording both prompt finalists'
stage_outputs=execute_matrix([{'prompt_id':prompt_id,'context_depth':CONTEXT_DEPTH} for prompt_id in PROMPT_FINALISTS],screening_ids,f'context_depth_{CONTEXT_DEPTH}',judge_calibration_ids)


In [ ]:
def nested(value,path):
    for part in path.split('.'):
        if not isinstance(value,dict) or part not in value: return None
        value=value[part]
    return value
def judge_usage(run_dir):
    path=run_dir/'judge_results.jsonl'
    if not path.exists(): return {'input_tokens':0,'output_tokens':0,'total_tokens':0}
    batches={}
    for record in load_jsonl(path): batches[record.get('batch_id',record['question_id'])]=record.get('batch_usage',{})
    return {key:sum(batch.get(key,0) for batch in batches.values()) for key in ['input_tokens','output_tokens','total_tokens']}
rows=[]; score_sets={}
for report_path in sorted(RUNS_ROOT.glob('*/report.json')):
    report=json.loads(report_path.read_text()); config_path_local=report_path.parent/'tuning_config.json'
    if not config_path_local.exists(): continue
    tuning=json.loads(config_path_local.read_text()); generation_input=nested(report,'usage.input_tokens') or 0; generation_output=nested(report,'usage.output_tokens') or 0; judge_tokens=judge_usage(report_path.parent)
    row={**tuning,'run_dir':str(report_path.parent),'generation_success_rate':nested(report,'coverage.success_rate'),'judge_samples':nested(report,'ragas.n_samples'),'qa_f1':nested(report,'qa.f1'),'citation_f1':nested(report,'citations.citation_f1'),'citation_validity':nested(report,'citations.citation_validity'),'answer_correctness':nested(report,'ragas.answer_correctness'),'faithfulness':nested(report,'ragas.faithfulness'),'answer_relevancy':nested(report,'ragas.answer_relevancy'),'generation_input_tokens':generation_input,'generation_output_tokens':generation_output,'judge_input_tokens':judge_tokens['input_tokens'],'judge_output_tokens':judge_tokens['output_tokens'],'generation_cost_usd':generation_input/1e6*GENERATOR_INPUT_PER_MILLION_USD+generation_output/1e6*GENERATOR_OUTPUT_PER_MILLION_USD,'judge_cost_usd':judge_tokens['input_tokens']/1e6*JUDGE_INPUT_PER_MILLION_USD+judge_tokens['output_tokens']/1e6*JUDGE_OUTPUT_PER_MILLION_USD,'generation_p95_ms':nested(report,'latency.llm.p95_ms'),'total_p95_ms':nested(report,'latency.total.p95_ms')}
    rows.append(row)
    score_sets[str(report_path.parent)]={record['question_id']:record for record in load_jsonl(report_path.parent/'deterministic_scores.jsonl')}
comparison=pd.DataFrame(rows)
if not comparison.empty:
    comparison.to_csv(RESULTS/'phase2b_comparison.csv',index=False); display(comparison.sort_values(['stage','answer_correctness'],ascending=[True,False]))
def cluster_paired_delta(left,right,metric,samples=1000):
    pairs=[]
    for qid in sorted(set(left)&set(right)):
        lv=nested(left[qid],metric); rv=nested(right[qid],metric)
        if lv is not None and rv is not None: pairs.append((left[qid]['article_key'],float(rv)-float(lv)))
    if not pairs: return None
    articles={}
    for article,delta in pairs: articles.setdefault(article,[]).append(delta)
    article_means=[sum(values)/len(values) for values in articles.values()]; import random
    rng=random.Random(SEED); boot=sorted(sum(rng.choice(article_means) for _ in article_means)/len(article_means) for _ in range(samples))
    return {'n_pairs':len(pairs),'n_articles':len(article_means),'mean_delta_right_minus_left':sum(delta for _,delta in pairs)/len(pairs),'article_macro_delta':sum(article_means)/len(article_means),'ci95_low':boot[int(.025*(samples-1))],'ci95_high':boot[int(.975*(samples-1))]}
paired=[]
if not comparison.empty:
    import itertools
    for stage,group in comparison.groupby('stage'):
        records=group.to_dict('records')
        for left,right in itertools.combinations(records,2):
            for metric in ['ragas.answer_correctness','ragas.faithfulness','citations.citation_f1']:
                result=cluster_paired_delta(score_sets[left['run_dir']],score_sets[right['run_dir']],metric)
                if result: paired.append({'stage':stage,'left':f"{left['prompt_id']}_d{left['context_depth']}",'right':f"{right['prompt_id']}_d{right['context_depth']}",'metric':metric,**result})
paired_frame=pd.DataFrame(paired)
if not paired_frame.empty: paired_frame.to_csv(RESULTS/'phase2b_paired_comparisons.csv',index=False); display(paired_frame)
run_manifest={'schema_version':1,'stage':RUN_STAGE,'repo_commit':REPO_COMMIT,'artifact_repo':HF_ARTIFACT_REPO_ID,'artifact_revision':HF_ARTIFACT_REVISION,'artifact_sha256':HF_ARTIFACT_SHA256,'baseline_fingerprint':baseline_manifest['run_fingerprint'],'prompt_registry_sha256':sha256_file(PROJECT_ROOT/'configs/experiments/phase2_generation_prompts.yaml'),'generator':{'model':GENERATOR_MODEL,'reasoning_effort':GENERATOR_REASONING_EFFORT,'max_tokens':GENERATOR_MAX_TOKENS},'judge':{'model':JUDGE_MODEL,'reasoning_effort':JUDGE_REASONING_EFFORT,'max_tokens':JUDGE_MAX_TOKENS},'prompt_finalists':PROMPT_FINALISTS,'finalist_configs':FINALIST_CONFIGS,'locked_winner':LOCKED_WINNER,'heldout_approved':HELDOUT_APPROVED,'generated_at':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())}
write_json(RESULTS/f'{RUN_STAGE}_manifest.json',run_manifest)
checkpoint=write_checkpoint()
result_bundle=shutil.make_archive(str(RUNTIME_ROOT/f'phase2b_{RUN_STAGE}_results'),'zip',root_dir=RESULTS)
print('Checkpoint:',checkpoint); print('Results:',result_bundle)


## 4. Export

Download or retain both the result bundle and checkpoint. Later stages must be configured from reviewed result artifacts, never by changing subset IDs.


In [ ]:
upstream={'path':UPSTREAM_DECISION_PATH or None,'sha256':sha256_file(UPSTREAM_DECISION_PATH) if UPSTREAM_DECISION_PATH and Path(UPSTREAM_DECISION_PATH).exists() else None}
write_json(RESULTS/'upstream_decision.json',upstream)
checkpoint=write_checkpoint()
result_bundle=RUNTIME_ROOT/f'{RUN_ID}_results.zip'; temporary=result_bundle.with_suffix('.zip.tmp')
with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for name in ['runs','question_ids','prompts','results','logs','heldout_trace']:
        root=WORK_ROOT/name
        if root.exists():
            for path in root.rglob('*'):
                if path.is_file(): archive.write(path,path.relative_to(WORK_ROOT))
temporary.replace(result_bundle)
destination=Path('/content/drive/MyDrive/newsqa_phase2b'); destination.mkdir(parents=True,exist_ok=True)
for artifact in [Path(checkpoint),Path(result_bundle)]:
    target=destination/artifact.name
    if artifact.resolve()!=target.resolve(): shutil.copy2(artifact,target)
    print('Saved:',target,round(target.stat().st_size/2**20,1),'MiB')
